# Temporal relational GNNs: first feasibility checks

Start on CPU. This notebook runs the current synthetic implementation, records its environment,
and saves logs to Drive. It does **not** train a RelBench model or establish an end-to-end speedup.
The optional GPU stage validates CUDA execution. The optional sampler stage checks a tiny PyG graph.

Use a paid GPU only when you reach those experiments. Runtimes can disappear; rerun setup after
reconnecting. Keep active datasets on the VM disk and persistent results/checkpoints in Drive.

The public repository clones without a GitHub token or Colab secret. Leave
`USE_GITHUB_TOKEN = False` in setup. For a private historical copy only, set it to `True`
and enable a read-only `GITHUB_TOKEN` secret with **Contents: read** for that repository.
Do not paste a token into a cell, output, Git URL, or GitHub commit.

In [ ]:
import os
import subprocess
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path("/content/temporal-relational-gnn")
PROJECT_REF = "main"  # Replace with a recorded commit for a later reproduction.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Public cloning needs no Colab secret. Enable only for a private historical copy.
USE_GITHUB_TOKEN = False

token = None
clone_environment = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
for inherited_name in ("GIT_ASKPASS", "SSH_ASKPASS", "TRGNN_CLONE_TOKEN"):
    clone_environment.pop(inherited_name, None)
try:
    with tempfile.TemporaryDirectory() as temporary:
        if USE_GITHUB_TOKEN:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN")
            askpass = Path(temporary) / "askpass.py"
            askpass.write_text(
                "#!/usr/bin/env python3\n"
                "import os, sys\n"
                'print("x-access-token" if "username" in sys.argv[1].lower() '
                'else os.environ["TRGNN_CLONE_TOKEN"])\n'
            )
            askpass.chmod(0o700)
            clone_environment.update({"GIT_ASKPASS": str(askpass), "TRGNN_CLONE_TOKEN": token})
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        subprocess.run(command, env=clone_environment, check=True)
finally:
    token = None
    clone_environment.pop("TRGNN_CLONE_TOKEN", None)
    clone_environment.pop("GIT_ASKPASS", None)
subprocess.run(["git", "checkout", PROJECT_REF], cwd=PROJECT_ROOT, check=True)
if PROJECT_REF == "main":
    subprocess.run(["git", "merge", "--ff-only", "origin/main"], cwd=PROJECT_ROOT, check=True)
subprocess.run(["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, check=True)

Install the project and test tools. Keep Colab's existing compatible PyTorch/CUDA build; exact
versions are captured in every run. PyG is pinned by the project. Reproductions must use the same
recorded environment. If installation changes packages already imported, restart and rerun setup.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    check=True,
)

Choose persistent storage. Mounting Drive requires your authorization. With `USE_DRIVE=False`,
outputs are temporary and must be downloaded before the VM is deleted. The budget fields are
manual observations from Colab's resource panel; they are not estimated GPU prices.

In [ ]:
import json
import uuid
from datetime import datetime, timezone

USE_DRIVE = True
COMPUTE_UNITS_PER_HOUR = None  # Copy the displayed rate for the active runtime.
COMPUTE_UNITS_REMAINING = None

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    persistent_root = Path("/content/drive/MyDrive/temporal-relational-gnn")
else:
    persistent_root = Path("/content/temporary-trgnn")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
RUN_DIR = persistent_root / "runs" / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "colab-budget.json").write_text(
    json.dumps(
        {
            "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
            "units_remaining_observed": COMPUTE_UNITS_REMAINING,
            "planned_total_units": 200,
            "reserve_units": 50,
        },
        indent=2,
    )
)
print(f"Persistent run directory: {RUN_DIR}")


def run_logged(arguments, log_name):
    result = subprocess.run(arguments, cwd=PROJECT_ROOT, capture_output=True, text=True)
    (RUN_DIR / log_name).write_text(result.stdout + "\n" + result.stderr)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()

Run CPU correctness first. A CUDA test is explicitly skipped on CPU; that skip is not GPU validation.
The suite includes equivalence of seed outputs, gradients, and an optimizer step, plus difficult
relationship and occurrence cases. No full dataset download is involved.

In [ ]:
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")
run_logged(
    [
        sys.executable,
        "-m",
        "trgnn.preflight",
        "--output",
        str(RUN_DIR / "preflight-cpu.json"),
    ],
    "preflight-cpu.log",
)
run_logged(
    [
        sys.executable,
        "-m",
        "trgnn.benchmark",
        "--device",
        "cpu",
        "--batch-size",
        "8",
        "--channels",
        "64",
        "--fanouts",
        "4",
        "2",
        "--output",
        str(RUN_DIR / "synthetic-cpu.json"),
    ],
    "synthetic-cpu.log",
)

Fetch the pinned public schema and task manifests (small files only). This is an initial source
check, not a complete task/feature audit or a full database download.

In [ ]:
run_logged(
    [
        sys.executable,
        "-m",
        "trgnn.dataset_manifest",
        "--output",
        str(RUN_DIR / "dataset-manifests.json"),
    ],
    "dataset-manifests.log",
)

## Optional CUDA stage

Select a GPU under **Runtime → Change runtime type**, rerun setup if the VM resets, and set
`RUN_GPU=True`. Record the newly displayed consumption rate. This stage requires an NVIDIA GPU;
it must fail instead of silently measuring CPU. Compare all modes within the same session.

In [ ]:
RUN_GPU = False

if RUN_GPU:
    run_logged(
        [
            sys.executable,
            "-m",
            "trgnn.preflight",
            "--require-cuda",
            "--output",
            str(RUN_DIR / "preflight-gpu.json"),
        ],
        "preflight-gpu.log",
    )
    run_logged([sys.executable, "-m", "pytest", "-q"], "correctness-gpu.log")
    run_logged(
        [
            sys.executable,
            "-m",
            "trgnn.benchmark",
            "--device",
            "cuda",
            "--batch-size",
            "32",
            "--channels",
            "64",
            "--fanouts",
            "16",
            "8",
            "--output",
            str(RUN_DIR / "synthetic-gpu.json"),
        ],
        "synthetic-gpu.log",
    )
else:
    print("CUDA stage not run. No GPU result has been established.")

## Optional actual temporal-sampler check

The graph-core prototype does not require compiled extensions, but PyG's temporal sampling does.
This installs a binary `pyg-lib` wheel for the active PyTorch/CUDA build; no source build fallback
is attempted. A missing compatible wheel is a feasibility issue to resolve before loading real data.

In [ ]:
CHECK_TEMPORAL_SAMPLER = False

if CHECK_TEMPORAL_SAMPLER:
    run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
    run_logged(
        [
            sys.executable,
            "-m",
            "trgnn.preflight",
            "--check-sampler",
            "--output",
            str(RUN_DIR / "sampler.json"),
        ],
        "sampler.log",
    )
else:
    print("Actual temporal sampler not checked.")

## Next milestone

Read `docs/PLAN.md`. Integrate the actual historical RelBench graph and one real sampled batch;
retain the original feature/time encoders and head. Verify identical logits across modes, record
actual RAM/VRAM use, and profile the complete pipeline before training the final checkpoint.
The current synthetic notebook does not complete that milestone.

Save your notebook, confirm logs exist in Drive, and disconnect/delete the runtime when done.